In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [35]:
def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def solvenominal (sets,p,R,r,m,r_f,c,ordering):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    a = cp.Variable(I)
    constraints = [a>= 0, a<=1, cp.sum(a)<= 1]
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

In [68]:
def cut_plane(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [a>= 0, a<=1, cp.sum(a)<= 1]
    h = np.zeros(N)
    iterations = 1
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    obj_value = prob.value
    [rbvalue,q_b] = robustcheck(w,R,r,p,m,r_f)
    nonstop = True
    while nonstop:
        print(rbvalue)
        if rbvalue <= c+1e-5:
            return(w,obj_value,iterations)
        h = q_b
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c+1e-5)
        obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        [rbvalue,q_b] = robustcheck(w,R,r,p,m,r_f)
        iterations = iterations + 1
        print(iterations)
    

In [53]:
np.random.seed(5)

In [69]:
N=10
p = np.zeros(N)+1/N
I = 20
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))

[-0.0477504   0.08489416  0.0890097   0.12804653  0.09193614  0.00045576
 -0.01882926  0.09144077  0.12054145  0.06722794  0.07052161 -0.01652886
 -0.01560558  0.01522542  0.05975289 -0.05272725  0.06070858  0.08242678
 -0.05677544 -0.0338457 ]


In [70]:
r = 0.3
m = 0.95    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.2
print(cut_plane(R,r,c,p,m,r_f))

0.09582204561349686
(array([0.00000000e+00, 1.00995628e-13, 1.70775685e-13, 1.00000000e+00,
       2.44475198e-13, 0.00000000e+00, 0.00000000e+00, 2.30241228e-13,
       1.17746928e-12, 1.27380043e-14, 1.56698601e-14, 0.00000000e+00,
       0.00000000e+00, 7.28145498e-14, 1.52078904e-14, 0.00000000e+00,
       1.44062056e-14, 7.27544408e-14, 0.00000000e+00, 0.00000000e+00]), 0.12804653069213964, 1)


In [23]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
psets

[[0], [1], [2], [0, 1], [0, 2], [1, 2], [0, 1, 2]]

In [34]:
h = np.zeros(N)
    
for i in range(N-1):
    h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
h[N-1]=h_3(p[N-1],m)

for ind in psets:
     print(sum(h[ind])>h_3(sum(p[ind]),m))

False
False
False
False
False
False
False
